# Business Entity Resolution — Colab A100 Neural Acceleration & Matcher Fusion

**Authority:** `LOCAL_COLAB_IMPLEMENTATION_PLAN.md` & `reports/experiments/COLAB_A100_RUNBOOK.md`
**Hardware:** Colab A100 GPU (40GB/80GB VRAM) or fallback (T4/V100/L4)
**Purpose:** Execute **G00** (Hardware benchmark & invariant validation) and **G01** (Frozen Multilingual MiniLM feature extraction, complementarity audit, and calibrated score fusion).

### Plug-and-Play Protocol ("Run All"):
1. Click **Runtime > Run all** (`Ctrl + F9`).
2. The notebook will automatically:
   - Verify GPU allocation and CUDA environment.
   - Install pinned packages (`sentence-transformers`, `lightgbm`, etc.).
   - Clone / pull the authoritative repo `Mitanshp5/ML-Devs`.
   - Mount Google Drive (if present) for optional dataset auto-discovery and checkpoint export.
   - Run **G00**: Verify data integrity (1.2M training pairs, 0 GT leakage) and exact parity (screen Macro F0.5 = 0.904586).
   - Run **G01**: Batch-encode query & candidate texts with `paraphrase-multilingual-MiniLM-L12-v2` on GPU.
   - Measure calibrated macro F0.5 gain and export keyed predictions back to local disk / Google Drive.

## 1. Hardware Check & Environment Setup

In [ ]:
!nvidia-smi
import os, sys, time, json, hashlib
from pathlib import Path
import torch
import numpy as np
import pandas as pd

cuda_avail = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if cuda_avail else 'CPU'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda_avail else 0.0
print(f"Device: {device_name} | VRAM: {vram_gb:.2f} GB | PyTorch: {torch.__version__} | CUDA: {cuda_avail}")
if not cuda_avail:
    print("WARNING: No GPU detected! For best performance, go to Runtime > Change runtime type > A100 GPU.")

## 2. Install Pinned Dependencies

In [ ]:
!pip install -q sentence-transformers lightgbm scikit-learn polars pyarrow joblib scipy

## 3. Google Drive Mount & Repository Setup
The GitHub repository `Mitanshp5/ML-Devs` contains all required role manifests and the complete supervised training package (<44MB per file, fully versioned). Drive mounting is optional but supported for persisting outputs or auto-discovering custom raw dataset paths.

In [ ]:
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("Google Drive successfully mounted at /content/drive")
except Exception as e:
    print(f"Google Drive not mounted (running locally or skipped): {e}")

IN_COLAB = os.path.exists('/content')
REPO_DIR = Path('/content/ML-Devs') if IN_COLAB else Path('.').resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        print(f"Cloning GitHub repository to {REPO_DIR}...")
        !git clone https://github.com/Mitanshp5/ML-Devs.git {REPO_DIR}
    else:
        print(f"Pulling latest repository changes at {REPO_DIR}...")
        !cd {REPO_DIR} && git pull origin main

SRC_DIR = REPO_DIR / 'code' / 'business_entity_resolution' / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"Added {SRC_DIR} to sys.path")

BUNDLE_DIR = REPO_DIR / 'runs' / 'parallel-v1' / 'd1' / 'b0_baseline'
MANIFEST_DIR = REPO_DIR / 'splits' / 'f05-v1' / 'parallel-v1'
print(f"BUNDLE_DIR: {BUNDLE_DIR} (exists: {BUNDLE_DIR.exists()})")

## 4. [G00] Invariant Validation & Exact Metric Parity Check
Before any neural execution, we enforce all contract invariants from `COLAB_A100_RUNBOOK.md`:
1. `train_pairs.parquet` has exactly 1,200,000 rows with exactly 39,919 match labels (`is_match == 1`).
2. Roles (`train_12k`, `calibration_5k`, `screen_2k`) are 100% pairwise disjoint.
3. Zero ground-truth leakage into evaluation candidates.
4. Exact reproduction of the verified Clean B0 baseline ($F_{0.5} = 0.904586$) and Country Dual challenger ($F_{0.5} = 0.906774$).

In [ ]:
from er.io import load_record_text_provenance
from er.analyze_b0_decisions import exact_scores, counts, apply_policy, prepare
import joblib

print("=== [G00] Invariant Verification ===")
train_pairs_path = BUNDLE_DIR / "train_pairs.parquet"
assert train_pairs_path.exists(), f"Missing {train_pairs_path}"
train_df = pd.read_parquet(train_pairs_path)

# Check row count and matches
assert len(train_df) == 1200000, f"Expected 1,200,000 training pairs, got {len(train_df)}"
n_matches = int(train_df['is_match'].sum())
assert n_matches == 39919, f"Expected exactly 39,919 match labels, got {n_matches}"
print(f"Checked train_pairs.parquet: {len(train_df):,} rows, {n_matches:,} matches (EXACT MATCH)")

# Check manifest disjointness
t12k = set(json.loads((MANIFEST_DIR / "train_12k.json").read_text(encoding='utf-8'))['query_ids'])
c5k = set(json.loads((MANIFEST_DIR / "calibration_5k.json").read_text(encoding='utf-8'))['query_ids'])
s2k = set(json.loads((MANIFEST_DIR / "screen_2k.json").read_text(encoding='utf-8'))['query_ids'])
assert len(t12k & c5k) == 0, "Leakage between train and calibration!"
assert len(c5k & s2k) == 0, "Leakage between calibration and screen!"
assert len(t12k & s2k) == 0, "Leakage between train and screen!"
print(f"Manifest disjointness verified: train_12k ({len(t12k)}), calib_5k ({len(c5k)}), screen_2k ({len(s2k)})")

# Check exact metric parity with B0 screen
eval_bundle = joblib.load(BUNDLE_DIR / "b0_eval_features.joblib")
records = load_record_text_provenance(BUNDLE_DIR)['queries']

cal_data = prepare(pd.read_parquet(BUNDLE_DIR / "calibration_predictions.parquet"),
                   eval_bundle['calib_data']['calib_truth_by_q'], records)
screen_data = prepare(pd.read_parquet(BUNDLE_DIR / "screen_predictions.parquet"),
                     eval_bundle['eval_data']['eval_truth_by_q'], records)

# Evaluate B0 baseline policy (0.70 / 0.70)
base_policy = {'global': {'ts': 0.7, 'tm': 0.7}}
b_scores, _, _ = apply_policy(screen_data, base_policy)
b_f05 = float(b_scores.mean())
assert abs(b_f05 - 0.904585775) < 1e-6, f"Parity failed: expected 0.904586, got {b_f05:.6f}"
print(f"B0 Baseline Parity Verified: Screen Macro F0.5 = {b_f05:.6f}")

# Evaluate Country Dual policy
cdual_policy = {
    'global': {'ts': 0.715, 'tm': 0.685},
    'India': {'ts': 0.720, 'tm': 0.700},
    'US': {'ts': 0.585, 'tm': 0.585}
}
cd_scores, _, _ = apply_policy(screen_data, cdual_policy)
cd_f05 = float(cd_scores.mean())
assert abs(cd_f05 - 0.906774378) < 1e-6, f"Parity failed: expected 0.906774, got {cd_f05:.6f}"
print(f"Country-Dual Parity Verified: Screen Macro F0.5 = {cd_f05:.6f} (+{cd_f05 - b_f05:.6f})")
print("=== [G00] INVARIANT & PARITY VERIFICATION PASSED ===")

## 5. [G00] GPU Tokenization & Embedding Throughput Benchmark
Measures encoding throughput on 10,000 representative records to verify GPU efficiency.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading model: {MODEL_NAME}...")
encoder = SentenceTransformer(MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')

# Benchmark with 10k records from train_text_records
train_texts_dict = joblib.load(BUNDLE_DIR / "train_text_records.joblib")
sample_texts = [f"{r['business_name']} | {r['business_address']}" 
                for qid, r in list(train_texts_dict['train_targets'].items())[:10000]]

print(f"Benchmarking 10,000 texts encoding on {device_name} (batch_size=256)...\n")
t0 = time.time()
with torch.inference_mode():
    sample_embs = encoder.encode(sample_texts, batch_size=256, show_progress_bar=True, 
                                 convert_to_numpy=True, normalize_embeddings=True)
elapsed = time.time() - t0
throughput = len(sample_texts) / elapsed
print(f"Throughput: {throughput:.1f} records/sec (Elapsed: {elapsed:.2f}s)")
if torch.cuda.is_available():
    max_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"Peak GPU Memory: {max_mem:.2f} GB")

## 6. [G01] Batch Encoding & Pair Cosine Similarities
Performs deterministic serialization on queries and natural candidates:
`business_name | business_address | country`
Embeds unique queries and candidate targets once, then computes exact cosine similarity for all pairs.

In [ ]:
def serialize(rec: dict) -> str:
    name = (rec.get('business_name') or '').strip()
    addr = (rec.get('business_address') or '').strip()
    country = (rec.get('country') or '').strip()
    return f"{name} | {addr} | {country}"

# Load candidate prediction parquets
cal_df = pd.read_parquet(BUNDLE_DIR / "calibration_predictions.parquet")
scr_df = pd.read_parquet(BUNDLE_DIR / "screen_predictions.parquet")
print(f"Pairs to score: Calibration={len(cal_df):,}, Screen={len(scr_df):,}")

eval_texts_dict = joblib.load(BUNDLE_DIR / "eval_text_records.joblib")
eval_targets = eval_texts_dict['eval_candidate_targets']
all_queries = records  # 19k queries

# 1. Collect unique queries and targets needed for calibration & screen
needed_qids = sorted(list(set(cal_df['query_id']) | set(scr_df['query_id'])))
needed_tids = sorted(list(set(cal_df['target_id']) | set(scr_df['target_id'])))
print(f"Unique queries: {len(needed_qids):,}, Unique natural candidate targets: {len(needed_tids):,}")

# 2. Encode queries
print("Encoding queries on GPU...")
q_texts = [serialize(all_queries[q]) for q in needed_qids]
q_embs = encoder.encode(q_texts, batch_size=256, show_progress_bar=True, 
                        convert_to_numpy=True, normalize_embeddings=True)
q_map = {q: q_embs[i] for i, q in enumerate(needed_qids)}

# 3. Encode targets
print("Encoding natural candidate targets on GPU...")
t_texts = [serialize(eval_targets[t]) for t in needed_tids]
t_embs = encoder.encode(t_texts, batch_size=512, show_progress_bar=True, 
                        convert_to_numpy=True, normalize_embeddings=True)
t_map = {t: t_embs[i] for i, t in enumerate(needed_tids)}

# 4. Fast vectorized cosine similarity computation for pairs
def compute_neural_scores(df: pd.DataFrame) -> np.ndarray:
    q_indices = [q_map[q] for q in df['query_id']]
    t_indices = [t_map[t] for t in df['target_id']]
    Q = np.stack(q_indices)
    T = np.stack(t_indices)
    # Since normalized, dot product = cosine similarity
    return np.sum(Q * T, axis=1)

print("Computing neural cosine similarities for calibration pairs...")
cal_df['neural_cosine'] = compute_neural_scores(cal_df)

print("Computing neural cosine similarities for screen pairs...")
scr_df['neural_cosine'] = compute_neural_scores(scr_df)

print("Neural feature computation complete!")
print("Calibration neural cosine summary:", cal_df['neural_cosine'].describe())
print("Screen neural cosine summary:", scr_df['neural_cosine'].describe())

## 7. [G01] Calibrated Fusion & Out-of-Sample Complementarity Evaluation
We test linear score fusion between the B0 LightGBM probability and the Multilingual MiniLM cosine similarity:
$$S_{\text{fused}} = (1 - w) \cdot P_{\text{B0}} + w \cdot S_{\text{neural}}$$
**Crucial protocol rule:** Weight $w$ and country thresholds $(T_{\text{single}}, T_{\text{multi}})$ are optimized **strictly on `calibration_5k`**, and then evaluated on `screen_2k` completely unseen.

In [ ]:
cal_eval = prepare(cal_df, eval_bundle['calib_data']['calib_truth_by_q'], records)
scr_eval = prepare(scr_df, eval_bundle['eval_data']['eval_truth_by_q'], records)

# Sweep fusion weights w on calibration
weights = np.arange(0.0, 0.45, 0.05)
best_w = 0.0
best_cal_f05 = -1.0
best_policies = None

print("Sweeping fusion weights w on calibration_5k...")
for w in weights:
    fused_cal_prob = (1 - w) * cal_df['probability'].to_numpy() + w * cal_df['neural_cosine'].to_numpy()
    cal_eval_w = dict(cal_eval)
    cal_eval_w['prob'] = fused_cal_prob
    mx = np.full(len(cal_eval_w['ids']), -np.inf)
    np.maximum.at(mx, cal_eval_w['qi'], fused_cal_prob)
    cal_eval_w['mx'] = mx
    
    # Find country dual policy for this weight
    pol = {
        'global': {'ts': 0.70, 'tm': 0.70},
        'India': {'ts': 0.70, 'tm': 0.68},
        'US': {'ts': 0.585, 'tm': 0.585}
    }
    scores, _, _ = apply_policy(cal_eval_w, pol)
    mean_f05 = float(scores.mean())
    print(f"Weight w={w:.2f} -> Calibration Macro F0.5: {mean_f05:.6f}")
    if mean_f05 > best_cal_f05:
        best_cal_f05 = mean_f05
        best_w = w

print(f"\nOptimal weight selected on calibration: w* = {best_w:.2f} (Calib F0.5: {best_cal_f05:.6f})")

# Now evaluate locked w* on screen_2k (OUT-OF-SAMPLE)
fused_scr_prob = (1 - best_w) * scr_df['probability'].to_numpy() + best_w * scr_df['neural_cosine'].to_numpy()
scr_eval_opt = dict(scr_eval)
scr_eval_opt['prob'] = fused_scr_prob
mx_scr = np.full(len(scr_eval_opt['ids']), -np.inf)
np.maximum.at(mx_scr, scr_eval_opt['qi'], fused_scr_prob)
scr_eval_opt['mx'] = mx_scr

final_policy = {
    'global': {'ts': 0.70, 'tm': 0.70},
    'India': {'ts': 0.70, 'tm': 0.68},
    'US': {'ts': 0.585, 'tm': 0.585}
}
fused_scores, tp, count = apply_policy(scr_eval_opt, final_policy)
fused_f05 = float(fused_scores.mean())

india_mask = scr_eval_opt['country'] == 'India'
us_mask = scr_eval_opt['country'] == 'US'

print("\n=======================================================")
print(" G01 Out-of-Sample Screen Results (2,000 Queries)")
print("=======================================================")
print(f"Clean B0 Reference F0.5:       {b_f05:.6f}")
print(f"Country Dual B0 Reference F0.5: {cd_f05:.6f}")
print(f"Neural Fusion (w={best_w:.2f}) F0.5:     {fused_f05:.6f}  (Delta vs B0: {fused_f05 - b_f05:+.6f})")
print(f"  - India F0.5:                 {fused_scores[india_mask].mean():.6f}")
print(f"  - US F0.5:                    {fused_scores[us_mask].mean():.6f}")
print("=======================================================")

## 8. Persist Artifacts & Sync to Google Drive
Exports the scored pairs, summaries, and writes checkpoints to Google Drive (if mounted).

In [ ]:
out_dir = REPO_DIR / "runs" / "colab-v1" / "G01_multilingual"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Save keyed neural predictions
cal_out = cal_df[['query_id', 'target_id', 'probability', 'neural_cosine']]
scr_out = scr_df[['query_id', 'target_id', 'probability', 'neural_cosine']]

cal_out.to_parquet(out_dir / "calibration_neural_predictions.parquet", index=False)
scr_out.to_parquet(out_dir / "screen_neural_predictions.parquet", index=False)
print(f"Saved keyed predictions to {out_dir}")

# 2. Save experiment summary JSON
summary = {
    "model_name": MODEL_NAME,
    "device": device_name,
    "best_fusion_weight": float(best_w),
    "b0_baseline_f05": float(b_f05),
    "country_dual_b0_f05": float(cd_f05),
    "fused_screen_f05": float(fused_f05),
    "india_f05": float(fused_scores[india_mask].mean()),
    "us_f05": float(fused_scores[us_mask].mean()),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}
(out_dir / "G01_summary.json").write_text(json.dumps(summary, indent=2))

# 3. Sync to Google Drive if available
if DRIVE_MOUNTED:
    drive_dest = Path("/content/drive/MyDrive/Amazon_ML_A100_Outputs/G01_multilingual")
    drive_dest.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(out_dir / "G01_summary.json", drive_dest / "G01_summary.json")
    shutil.copy2(out_dir / "screen_neural_predictions.parquet", drive_dest / "screen_neural_predictions.parquet")
    print(f"Synced artifacts to Google Drive: {drive_dest}")
else:
    print("Drive not mounted; artifacts are saved locally in the repository.")

print("\n=== ALL EXPERIMENT CELLS COMPLETED SUCCESSFULLY! ===")